In [0]:
# 1. Setup your paths using your Unity Catalog Volume
from pyspark.sql.functions import current_timestamp, col

def ingest_json_bronze(file_name, target_table):
    """
    Ingests JSON data from Azure Data Lake using Auto Loader (PySpark).
    """
    base_path = "/Volumes/f1_catalognew/f1_schema/landing_zone"

    # 1. READ (The Streaming Part)
    raw_stream = (spark.readStream
                  .format("cloudFiles")
                  .option("cloudFiles.format", "json")
                  .option("cloudFiles.schemaLocation", f"{base_path}/_metadata/schemas/{file_name}")
                  .option("cloudFiles.inferColumnTypes", "true")
                  .load(f"{base_path}/{file_name}.json"))

    # 2. TRANSFORM (Add Audit Columns - Essential for Debugging!)
    # This shows you care about data lineage.
    bronze_df = (raw_stream
                .withColumn("ingest_timestamp", current_timestamp())
                .withColumn("source_file", col("_metadata.file_path")))

    # 3. WRITE (The Delta Part)
    return (bronze_df.writeStream
            .format("delta")
            .option("checkpointLocation", f"{base_path}/_metadata/checkpoints/{file_name}")
            .trigger(availableNow=True)
            .option("mergeSchema", "true")
            .outputMode("append")
            .toTable(target_table))
# Execute for both datasets
ingest_json_bronze("drivers", "f1_catalognew.f1_schema.drivers")
ingest_json_bronze("results", "f1_catalognew.f1_schema.results")

In [0]:
df = spark.read.json(
    "/Volumes/f1_catalognew/f1_schema/landing_zone/drivers.json/",
     
)
display(df)